# Reverse-Engineering Rank #5: THUNDER THUNDER, a Price-Adaptive Portfolio Farm

Most of this leaderboard runs one hardcoded action tape (see the long-running forum discussions on the fixed-strategy meta). The interesting agents are the handful that don't. This writeup reverse-engineers one of them: **THUNDER THUNDER**, currently **rank #5 at ~3124**, from 19 of their public ranked replays.

Everything here comes from **public data via the official Kaggle CLI** — the same replay files anyone can download. No private information, no scraping. Method and commands are at the bottom so you can reproduce or extend this for any team.

*Snapshot taken 2026-08-11. Top 5 at the time: Kaito Fukami 3220 · researchstudio.site 3135 · カワシギ 3130 · Syed Muhammad Gillani 3129 · THUNDER THUNDER 3124.*

## TL;DR

THUNDER THUNDER is **not a tape**. Across 19 sampled games it played **19 distinct builds** (cows 8–12, sheep 4–10, always 0 geese, 0 carrot/tomato). But the variation is not opponent-mirroring either — it is **price-driven portfolio sizing** on top of a fixed opening skeleton:

1. A **byte-identical modest opening** every game (it keeps cash in reserve on day 0 — no all-in).
2. Land on **exactly days 6 and 10**, three quadrants, never the fourth.
3. A **fertilizer race from day 2** — first sale of the game is FERTILIZER in slot 0.
4. Herd expansion **days 5–12, sized by realized product prices**, not by what the opponent builds.
5. A **day-20 wheat replant spike** feeding a ~250–285 unit wheat dump on days 24–29 — into a price its own season-long feed buying pushed up.
6. A crew of **14 hands/day from day 10** — it pays the hire cost curve most agents refuse to, and converts it into roughly 3,000 productive unit-actions per game.

Its cash curve never stalls mid-game ($8.6k–$21.8k at day 15 in every replay I deep-read, wins **and** losses), and its losses have a clear shape: mutual-glut games against other adaptive dairy farms when it over-expands.

## Sample

19 episodes from their strongest active submission (180+ episodes total): the 12 most recent plus 6 stride-sampled older ones, with 3 replays kept for turn-level reading (a win vs Tom3, losses vs Syed Muhammad Gillani and Lazarus Ogero).

Record in the sample: **9W–10L** — but the draw is almost entirely the current top table (Kaito Fukami, カワシギ, Gillani, Raoul Biagioni, Trex…). A 9–10 record against that draw is parity at the summit, not weakness. As always: never read a raw win rate without looking at who the games were against.

| ep | opponent | W/L | their build (c/s) | final $ (them) | final $ (opp) |
|---|---|---|---|---:|---:|
| 91462596 | Hajime Tamura | W | 10/4 | 140,965 | 99,635 |
| 92159538 | S. M. Gillani | L | 11/6 | 111,652 | 126,269 |
| 91294670 | [ODS] waniz | W | 9/4 | 111,074 | 90,758 |
| 92141679 | Trex | W | 8/10 | 107,251 | 93,282 |
| 92133998 | Raoul Biagioni | L | 9/5 | 105,304 | 107,005 |
| 92126500 | カワシギ | W | 9/5 | 94,675 | 92,618 |
| 92150166 | カワシギ | L | 10/4 | 86,546 | 93,361 |
| 91646245 | Teddy Tennant | W | 8/5 | 85,276 | 82,604 |
| 92133999 | Kaito Fukami | L | 10/5 | 82,413 | 92,025 |
| 91853249 | Ueddy | W | 10/7 | 81,668 | 81,208 |
| 92152110 | Tom3 | W | 10/7 | 77,617 | 75,528 |
| 92106963 | Raoul Biagioni | L | 8/6 | 73,756 | 74,606 |
| 92118981 | Kaito Fukami | L | 10/5 | 65,874 | 79,692 |
| 91317208 | Apollo's cattles | W | 10/8 | 64,831 | 53,111 |
| 92114369 | カワシギ | L | 8/7 | 59,220 | 63,041 |
| 92097432 | fistyee | L | 8/4 | 58,932 | 61,956 |
| 92159566 | Lazarus Ogero | L | 12/8 | 57,458 | 63,168 |
| 92106837 | S. M. Gillani | L | 11/8 | 56,967 | 61,340 |
| 91184073 | Matthew Labrador | W | 10/7 | 81,305 | 66,567 |

## The fixed skeleton

Three replays deep-read turn by turn; these parts were **identical in all three**:

**Day 0, hour 1** (the game's first market turn), the exact same basket every game:

```
HIRE ×4, BUY_ANIMAL COW 1, BUY_ANIMAL SHEEP 4,
BUY_SEED WHEAT 5, BUY_SEED MELON 5, BUY_PRODUCT WHEAT 5
```

Note what that is *not*: it is not the clone tape's day-0 all-in. It is sheep-first, small, and keeps a cash reserve. The herd is built later, from income.

- **BUY_LAND on day 6 and day 10.** Exactly those days, all three games. Three quadrants total; the fourth is never bought.
- **Day 2, first market turn: `SELL FERTILIZER 5` in slot 0** — before anything else that day. Fertilizer's market pool is shared, finite, and never refills; whoever sells it first captures the price. They race it from day 2 and keep selling it all game (~45 units by day 9).
- **Day 20: a wheat seed spike** — 44 / 46 / 48 seeds in the three games (vs a 1–8/day trickle otherwise). This matures into the endgame.
- **Hire schedule:** 4 hands on day 0, ~7 by day 7, then **14 per day from day 10 to the bell**. Hands re-hire daily at a steeply super-linear daily cost, and most of the field caps at ~10 because of it. THUNDER simply pays the cliff once income covers it — an income-sized crew, not a tile-sized one.

## The adaptive part: prices, not opponents

The obvious hypothesis — that it reads the opponent's pens and mirrors or counters — is **wrong**. Correlating their herd choices with the opponent's day-9 pen census across all 19 games gives nothing (|r| ≤ 0.22, n=19, i.e. noise).

What does correlate: **realized product prices around days 8–10**.

| signal | response | r |
|---|---|---:|
| WOOL price d8–10 | cows added after d10 | **+0.60** |
| WOOL price d8–10 | sheep added after d10 | +0.18 |
| MILK price d8–10 | cows added after d10 | −0.30 |
| opp d9 pens (any) | anything | ≤ ±0.22 |

The pattern reads as **portfolio sizing**: when premium animal-product prices are healthy in the mid-game window, it expands the herd hard (to ~10c/7s by day 12); when they're depressed, it holds a lean 8c/4s and leans on crops instead. On-board composition at day 9 is nearly always ~8c/4s — all the divergence happens after.

The one clearly directional case: vs **Trex**, the opponent abandoned sheep entirely (finished 9c/1s) while wool sat at $232 — and THUNDER played its only sheep-heavy build of the sample (7c/**10s**), taking the vacated wool market. Whether that's an explicit rule or just the price response doing its job, the effect is the same: it flows capital toward whatever the market is currently short of.

With n=19 I'd treat the exact mechanism as informed inference, not proof — but "prices in, composition out" fits every game in the sample, and "opponent pens in" fits none of it.

## The economy: six revenue lines, two of them unusual

Typical sold units per game: **WHEAT ~420 · STRAWBERRY ~270 · FERTILIZER ~250 · MILK ~225 · WOOL ~130 · MELON ~110.** Six lines, every game. Two deserve attention:

**Fertilizer is farmed, not incidental.** 368–382 `COLLECT_FERTILIZER` actions per game — 5.3% of their entire labor budget — yielding ~250–320 units sold plus ~62 `FERTILIZE` applications on their own crops. A zero-cost byproduct turned into a full revenue line, and (see above) raced from day 2 because the pool is first-come-first-served.

**Wheat is a triple-use self-hedge.** They seed ~146 wheat across the season *and* buy ~240 as feed, continuously, all game. The season-long buying does two jobs (feeds the herd, drains the shared wheat pool so the price climbs), and then the day-20 replant delivers ~250–285 units dumped on days 24–29 — **into the elevated price their own feed-buying created**. In the three deep-read games the d24–29 wheat dump was 266, 284 and 245 units, the single biggest endgame line each time.

Sell-slot ordering is computed per game, not a fixed table — first-slot product varies game to game, with wool/melon/strawberry tending early slots and wheat late. Consistent with impact-ordering (steep glut curves first), which forum discussions have previously measured as worth real money.

**Throughput:** ~7,150 unit-actions per game, 43% movement, ≈3,000 productive actions. For reference, the public clone tape executes ≈2,760 productive actions. The 14-hand crew is what services six simultaneous revenue lines.

**Cash curve (from the deep-read replays, day-15 midday):** $18.5k (W vs Tom3), $9.1k (L vs Gillani), $8.6k (L vs Ogero) — no mid-game cash stall even in losses. The day 10–15 income surge comes from the fertilizer race, early wool, melon's first harvest and the wheat trickle.

## Where it loses

- **Mutual-glut games against other adaptive dairy farms.** Its worst sampled game ($57.5k, vs Lazarus Ogero) was also its biggest herd (12c/8s): two price-reactive farms expand into the same premium products and crush the prices for both. The smaller farm loses less; over-expansion is the failure mode.
- **Low-price grinds.** When the mid-game price window reads depressed, it stays lean — and can be simply out-produced (the fistyee loss, both sides under $62k).
- Against the current top 4 in this sample it went 2–6 (Kaito 0–2, カワシギ 1–2, Gillani 0–2, + Biagioni 0–2). The summit is not solved; it is a knife fight between adaptive farms.

## What this says about the meta

The engine's current demand model punishes monoculture supply far more than earlier versions did — a fixed tape can glut its own market without any opponent trying to. The agents at the very top of the board now are the ones that (a) diversify across many lines, (b) read prices and size the portfolio accordingly, and (c) spend labor aggressively once income covers it. THUNDER THUNDER is a clean, legible example of that class: one fixed logistical skeleton, one adaptive capital-allocation layer on top.

## Reproduce it

All of this is public data via the official CLI — works for **any** team:

```bash
# leaderboard -> teamIds
kaggle competitions leaderboard kaggriculture -s --csv

# teamId -> their active submissions (works for any team)
kaggle competitions team-submissions <teamId> --format csv

# submissionId -> episodes
kaggle competitions episodes <submissionId> --format csv

# episodeId -> full replay JSON (~30 MB)
kaggle competitions replay <episodeId> -p replays/
```

In the replay JSON, `steps[t][p]["action"]` is a dict of `{"farmer": [...], "hands": [[...], ...], "market": [[...], ...]}` — and the **order of the `market` list is the slot order**, which is what makes sell-ordering readable from replays. `observation.farms` is public for both players; per-day censuses, prices and cash curves all reconstruct from it. Summarize each replay to a few KB and delete the raw file, or 30 MB × N will eat your disk.

Good hunting. And a genuine tip of the hat to THUNDER THUNDER — it is a beautifully engineered agent.

*— jeffmarcecadet*

In [ ]:
# Static analysis writeup - all measurement ran locally on replays downloaded
# via the official Kaggle CLI (commands in the section above).
print("See markdown above - reproduction commands included.")